In [0]:
import json
import time
import uuid
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import requests 
from pyspark.sql import functions as F 
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType

dbutils.widgets.text("catalog_name", "weather_cat_dev") 
CATALOG = dbutils.widgets.get("catalog_name") 

spark.sql(f"USE CATALOG {CATALOG}") 

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(ZoneInfo("Asia/Singapore"))

In [0]:
# Load Config
locations = [
    r.asDict()
    for r in spark.table(f"{CATALOG}.config.locations") 
    .filter("is_active = true")
    .collect()
]
params_dict = {
    r["param_key"]: r["param_value"]
    for r in spark.table(f"{CATALOG}.config.pipeline_params").collect() 
}

BASE_URL = params_dict["base_url"]
HOURLY_VARS = params_dict["hourly_vars"]
FORECAST_DAYS = params_dict["forecast_days"]
MAX_RETRIES = int(params_dict["max_retries"])
EXPECTED_SCHEMA = json.loads(params_dict["expected_hourly_schema"])

In [0]:
def detect_schema_drift(hourly_obj: dict, expected: dict) -> str:
    """Detects new fields, missing fields, and changed data types in hourly arrays."""
    try:
        if not isinstance(hourly_obj, dict):
            raise TypeError(f"'hourly_obj' must be a dict, got {type(hourly_obj).__name__}")
        if not isinstance(expected, dict):
            raise TypeError(f"'expected' must be a dict, got {type(expected).__name__}")

        changes = []
        actual_keys = set(hourly_obj.keys())
        expected_keys = set(expected.keys())

        new_fields = actual_keys - expected_keys
        missing_fields = expected_keys - actual_keys

        if new_fields:
            changes.append(f"NEW_FIELDS: {sorted(list(new_fields))}")
        if missing_fields:
            changes.append(f"MISSING_FIELDS: {sorted(list(missing_fields))}")

        for field in actual_keys.intersection(expected_keys):
            arr = hourly_obj.get(field)
            if isinstance(arr, list):
                # Find first non-null value to inspect runtime type
                sample = next((x for x in arr if x is not None), None)
                if sample is not None:
                    exp_type = expected[field]
                    if exp_type == "str" and not isinstance(sample, str):
                        changes.append(
                            f"TYPE_CHANGE({field}): expected str, got {type(sample).__name__}"
                        )
                    elif exp_type in ("float", "int") and isinstance(sample, str):
                        changes.append(
                            f"TYPE_CHANGE({field}): expected numeric, got str"
                        )

        return "; ".join(changes) if changes else "NONE"

    except Exception as exc:
        raise RuntimeError(f"Schema drift detection failed: {exc}") from exc

In [0]:
def fetch_with_retry(url: str, query_params: dict, max_retries: int = 3):
    """Handles HTTP 429 rate limits and 5xx transient server errors with backoff."""
    for attempt in range(max_retries + 1):
        try:
            resp = requests.get(url, params=query_params, timeout=15)
            if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
                time.sleep(2 ** (attempt + 1))
                continue
            return resp
        except requests.Timeout:
            if attempt == max_retries:
                raise
            time.sleep(2)

In [0]:
bronze_records = []
location_audits = []
succeeded_count = 0

for loc in locations:
    loc_id = loc["location_id"]
    city = loc["city_name"]
    q_params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "hourly": HOURLY_VARS,
        "forecast_days": FORECAST_DAYS,
        "timezone": "Asia/Singapore",
    }
    now_ts = datetime.now(ZoneInfo("Asia/Singapore"))

    try:
        resp = fetch_with_retry(BASE_URL, q_params,MAX_RETRIES)
        status_code = resp.status_code
        raw_text = resp.text
        req_url = resp.url

        # Always append raw response to Bronze regardless of downstream parsing
        bronze_records.append(
            (
                str(uuid.uuid4()),
                RUN_ID,
                now_ts,
                loc_id,
                city,
                req_url,
                status_code,
                raw_text,
            )
        )

        # Edge Case: Non-200 HTTP Status (400 Bad Request, 429, 5xx)
        if status_code != 200:
            raise ValueError(f"HTTP_{status_code}: {raw_text[:200]}")

        # Edge Case: Empty or cut-off JSON response
        if not raw_text.strip():
            raise ValueError("EMPTY_RESPONSE_BODY")
        payload = json.loads(raw_text)

        if "hourly" not in payload or not isinstance(payload["hourly"], dict):
            raise ValueError("MISSING_HOURLY_OBJECT")

        hourly = payload["hourly"]

        # Edge Case: Arrays of different lengths
        lengths = {
            k: len(v) for k, v in hourly.items() if isinstance(v, list)
        }
        if len(set(lengths.values())) > 1:
            raise ValueError(f"ARRAY_LENGTH_MISMATCH: {lengths}")

        row_count = next(iter(lengths.values()), 0)
        if row_count == 0:
            raise ValueError("ZERO_HOURLY_RECORDS")

        # Schema Handling: Log any new, missing, or type-changed fields
        drift_log = detect_schema_drift(hourly, EXPECTED_SCHEMA)

        succeeded_count += 1
        location_audits.append(
            (
                RUN_ID,
                loc_id,
                city,
                "SUCCESS",
                status_code,
                row_count,
                drift_log,
                None,
                now_ts,
            )
        )

    except Exception as exc:
        # Failure for one location is isolated; loop continues for remaining locations
        err_msg = str(exc)
        location_audits.append(
            (
                RUN_ID,
                loc_id,
                city,
                "FAILED",
                locals().get("status_code", -1),
                0,
                "N/A",
                err_msg,
                now_ts,
            )
        )

In [0]:
# Persist Bronze (Append-Only)
if bronze_records:
    bronze_schema = StructType(
        [
            StructField("ingestion_id", StringType(), False),
            StructField("run_id", StringType(), False),
            StructField("ingestion_ts", TimestampType(), False),
            StructField("location_id", StringType(), False),
            StructField("city_name", StringType(), True),
            StructField("request_url", StringType(), True),
            StructField("http_status", IntegerType(), True),
            StructField("raw_response", StringType(), True),
        ]
    )
    spark.createDataFrame(bronze_records, bronze_schema).write.mode( # type: ignore
        "append"
    ).saveAsTable(f"{CATALOG}.bronze.weather_raw")

In [0]:
# Persist Location Audits
if location_audits:
    loc_audit_schema = StructType(
        [
            StructField("run_id", StringType(), False),
            StructField("location_id", StringType(), True),
            StructField("city_name", StringType(), True),
            StructField("status", StringType(), True),
            StructField("http_status", IntegerType(), True),
            StructField("rows_extracted", IntegerType(), True),
            StructField("schema_changes", StringType(), True),
            StructField("error_message", StringType(), True),
            StructField("logged_at", TimestampType(), True),
        ]
    )
    spark.createDataFrame(location_audits, loc_audit_schema).write.mode( # type: ignore
        "append"
    ).saveAsTable(f"{CATALOG}.audit.location_audit")

In [0]:
# Store RUN_ID in task values for Silver & Gold tasks
dbutils.jobs.taskValues.set(key="run_id", value=RUN_ID) # type: ignore
dbutils.jobs.taskValues.set( # type: ignore
    key="start_time", value=START_TIME.isoformat()
)
dbutils.jobs.taskValues.set(key="locations_attempted", value=len(locations)) # type: ignore
dbutils.jobs.taskValues.set(key="locations_succeeded", value=succeeded_count)

In [0]:
dbutils.notebook.exit(
    f"Bronze Complete | Catalog: {CATALOG} | Run ID: {RUN_ID} | "
    f"Locations: {succeeded_count}/{len(locations)} "
)